# UPCoM Model

In [ ]:
from pycontrails.datalib.ecmwf import ERA5
from pycontrails.datalib.emis_grid import AEIC
from pycontrails.models.upcom import UPCOM, UPCOMParams
from pycontrails.models.humidity_scaling import ConstantHumidityScaling
from pycontrails.models.sac import SAC
from pycontrails.models.issr import ISSR

from datetime import datetime

# For visualization
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load Gridded Emissions Data
aeic = AEIC(
    time=("2019-01", "2019-12"),
    variables=["FUELBURN", "BC", "DISTANCE"],
    data_dir="/Users/gett613/Data/AEIC/AEIC_monthly",
    pressure_levels=[150, 400],
)
emis = aeic.open_metdataset()  # returns MetDataset
emis

In [ ]:
# Load meteorology data
time = (datetime(2022, 3, 1, 0), datetime(2022, 3, 1, 2))
variables = ["air_temperature", "specific_humidity"]
pressure_levels = [200, 250, 300]
era5 = ERA5(time, variables, pressure_levels)
met = era5.open_metdataset()
met

In [ ]:
# Create model with optional humidity scaling
scaling = ConstantHumidityScaling(rhi_adj=0.98)
model = UPCOM(met, humidity_scaling=scaling)

In [ ]:
# Run evaluation
result = model.eval()

In [ ]:
result

In [ ]:
# Access outputs
rhi = result["rhi"]
issr = result["issr"]
persistent_contrails = result["potential_persistent_contrail"]
temp = result["air_temperature"]

In [ ]:
# Get comparison from Native pycontrails routines (code from pycontrails/notebooks/SAC.ipynb and ISSR.ipynb)

# run model for across full input domain
# outputs global ice super-saturated regions as 1.0, all other regions as 0.0
issr_mds = ISSR(met, humidity_scaling=scaling).eval()
issr = issr_mds["issr"]

# evaluate SAC on met grid
sac_mds = SAC(met=met, humidity_scaling=scaling).eval()  # returns a MetDataset
sac = sac_mds["sac"]  # extract the SAC MetDataArray

# PPCF occurs where both the ISSR and SAC criteria are satisfied
ppcf = (issr.data == 1) & (sac.data == 1)

In [ ]:
sac.data.sel(time=result.data.time[0], level=250).transpose().plot(figsize=(12, 6))
plt.title('SAC from pycontrails at 250 hPa')
plt.show()

In [ ]:
ppcf.sel(time=result.data.time[0], level=250).transpose().plot(
    figsize=(12, 6), cbar_kwargs={"label": "PPCF (0/1)"}
)
plt.title('Potential persistent contrail formation at 250 hPa')
plt.show()

In [ ]:
issr

In [ ]:
result["issr"]

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

level = 300
time_value = result.data.time[0]
model_issr = result.data["issr"].sel(time=time_value, level=level)
reference_issr = issr.data.sel(time=time_value, level=level)
issr_difference = model_issr - reference_issr

fig, axes = plt.subplots(
    3, 1, figsize=(12, 15),
    subplot_kw={"projection": ccrs.Robinson(central_longitude=0)},
)
panel_data = [model_issr, reference_issr, issr_difference]
panel_titles = ["(a) UPCOM result['issr']", "(b) ISSR", "(c) Difference (UPCOM - ISSR)"]
panel_cmaps = ["YlOrRd", "YlOrRd", "RdBu_r"]
panel_limits = [(0, 1), (0, 1), (-1, 1)]
panel_cbar_labels = ["ISSR (0/1)", "ISSR (0/1)", "Difference (UPCOM - ISSR)"]

for ax, data, title, cmap, (vmin, vmax), cbar_label in zip(
    axes, panel_data, panel_titles, panel_cmaps, panel_limits, panel_cbar_labels
):
    data.transpose().plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_labels=False,
        cbar_kwargs={
            "orientation": "horizontal",
            "pad": 0.05,
            "shrink": 0.75,
            "label": cbar_label,
        },
    )
    ax.coastlines(linewidth=0.8)
    ax.gridlines(alpha=0.4, linestyle="--")
    ax.set_global()
    ax.set_title(title)

fig.suptitle(f"ISSR comparison at {level} hPa", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
#var='air_temperature'
var='potential_persistent_contrail'
#var='T_contr'
result.data[var].sel(time=result.data.time[0], level=250).transpose().plot(figsize=(12, 6))
plt.title(var+' at 250 hPa')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

level = 200
time_value = result.data.time[0]
model_ppcf = result.data["potential_persistent_contrail"].sel(
    time=time_value, level=level
)
reference_ppcf = ppcf.sel(time=time_value, level=level)
ppcf_difference = model_ppcf - reference_ppcf

fig, axes = plt.subplots(
    3, 1, figsize=(12, 15),
    subplot_kw={"projection": ccrs.Robinson(central_longitude=0)},
)
panel_data = [model_ppcf, reference_ppcf, ppcf_difference]
panel_titles = [
    "(a) UPCOM result['potential_persistent_contrail']",
    "(b) PPCF (ISSR & SAC)",
    "(c) Difference (UPCOM - PPCF)",
]
panel_cmaps = ["YlOrRd", "YlOrRd", "RdBu_r"]
panel_limits = [(0, 1), (0, 1), (-1, 1)]
panel_cbar_labels = [
    "Potential persistent contrail (0/1)",
    "PPCF (0/1)",
    "Difference (UPCOM - PPCF)",
]

for ax, data, title, cmap, (vmin, vmax), cbar_label in zip(
    axes, panel_data, panel_titles, panel_cmaps, panel_limits, panel_cbar_labels
):
    data.transpose().plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_labels=False,
        cbar_kwargs={
            "orientation": "horizontal",
            "pad": 0.05,
            "shrink": 0.75,
            "label": cbar_label,
        },
    )
    ax.coastlines(linewidth=0.8)
    ax.gridlines(alpha=0.4, linestyle="--")
    ax.set_global()
    ax.set_title(title)

fig.suptitle(f"PPCF comparison at {level} hPa", y=0.98)
plt.tight_layout()
plt.show()

### Plot

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, axes = plt.subplots(2, 2, figsize=(16, 8), 
                         subplot_kw={'projection': ccrs.PlateCarree()})

level = 300
time_idx = 1

fields = ['rhi', 'rh_liquid', 'issr', 'potential_persistent_contrail']
titles = ['RHi', 'RH (liquid)', 'ISSR', 'Persistent Contrails']

for ax, field, title in zip(axes.flat, fields, titles):
    data = result.data[field].sel(time=result.data.time[time_idx], level=level)
    data.transpose().plot(ax=ax, transform=ccrs.PlateCarree(), cbar_kwargs={'shrink': 0.8})
    ax.coastlines(color="white")
    ax.gridlines(alpha=0.8, linestyle='--')
    ax.set_title(f'{title} at {level} hPa')

plt.tight_layout()
plt.show()

### Not sure these work....

In [ ]:
### Uses built in xarray plotting

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
#import cartopy.feature as cfeature

fig, axes = plt.subplots(2, 2, figsize=(20, 14), 
                         subplot_kw={'projection': ccrs.Robinson()})

level = 250
time_idx = 0

fields = ['rhi', 'rh_liquid', 'issr', 'potential_persistent_contrail']
titles = ['Relative Humidity over Ice', 'Relative Humidity over Liquid', 
          'Ice Supersaturated Regions', 'Potential Persistent Contrails']
cmaps = ['RdYlBu_r', 'RdYlBu_r', 'YlOrRd', 'RdYlGn']
vmins = [0, 0, 0, 0]
vmaxs = [1.5, 1, 1, 1]

for ax, field, title, cmap, vmin, vmax in zip(axes.flat, fields, titles, cmaps, vmins, vmaxs):
    data = result.data[field].sel(time=result.data.time[time_idx], level=level)
    
    # Use xarray's plot with transform
    data.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap, vmin=vmin, vmax=vmax,
        cbar_kwargs={'orientation': 'horizontal', 'pad': 0.05, 
                     'shrink': 0.8, 'label': title}
    )
    
    ax.coastlines(linewidth=0.8)
#    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':', alpha=0.5)
    ax.gridlines(alpha=0.4, linestyle='--')
    ax.set_global()
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle(f'UPCOM Model Output at {level} hPa\n{result.data.time[time_idx].values}', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
#Uses contourf (matplotlib)
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
#import cartopy.feature as cfeature

fig, axes = plt.subplots(2, 2, figsize=(20, 14), 
                         subplot_kw={'projection': ccrs.Robinson(central_longitude=0)})

level = 250
time_idx = 0

fields = ['rhi', 'rh_liquid', 'issr', 'potential_persistent_contrail']
titles = ['Relative Humidity over Ice', 'Relative Humidity over Liquid', 
          'Ice Supersaturated Regions', 'Potential Persistent Contrails']
cmaps = ['RdYlBu_r', 'RdYlBu_r', 'YlOrRd', 'RdYlGn']
vmins = [0, 0, 0, 0]
vmaxs = [1.5, 1, 1, 1]

for ax, field, title, cmap, vmin, vmax in zip(axes.flat, fields, titles, cmaps, vmins, vmaxs):
    data = result.data[field].sel(time=result.data.time[time_idx], level=level)
    
    # Use contourf instead of pcolormesh
    im = ax.contourf(
        data.longitude, data.latitude, data.values,
        levels=20,  # Number of contour levels
        transform=ccrs.PlateCarree(),
        cmap=cmap, vmin=vmin, vmax=vmax,
        extend='both'
    )
    
    ax.coastlines(linewidth=0.8, color='black')
#    ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
#    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.2)
#    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':', alpha=0.5)
    ax.gridlines(alpha=0.4, linestyle='--')
    ax.set_global()
    
    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', 
                       pad=0.05, shrink=0.8, aspect=30)
    cbar.set_label(title, fontsize=10)
    
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

fig.suptitle(f'UPCOM Model Output at {level} hPa\n{result.data.time[time_idx].values}', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


### Diagnostics for Debugging

In [ ]:
import numpy as np

# Diagnostic code
print("=== Diagnostics ===")
print(f"T_contr type: {type(result.data['T_contr'])}")
print(f"T_contr shape: {result.data['T_contr'].shape}")
print(f"T_contr dims: {result.data['T_contr'].dims}")
print(f"T_contr min/max: {result.data['T_contr'].min().values:.2f} / {result.data['T_contr'].max().values:.2f}")

print(f"\nAir temp min/max: {result.data['air_temperature'].min().values:.2f} / {result.data['air_temperature'].max().values:.2f}")
print(f"RHi min/max: {result.data['rhi'].min().values:.3f} / {result.data['rhi'].max().values:.3f}")
print(f"RH_liquid min/max: {result.data['rh_liquid'].min().values:.3f} / {result.data['rh_liquid'].max().values:.3f}")

print(f"\nG min/max: {result.data['G'].min().values:.6f} / {result.data['G'].max().values:.6f}")
print(f"G > 0.053: {(result.data['G'] > 0.053).sum().values} points")

print(f"\nCondition checks (count of True):")
print(f"  RHi > 1.0: {(result.data['rhi'] > 1.0).sum().values}")
print(f"  T < T_contr: {(result.data['air_temperature'] < result.data['T_contr']).sum().values}")
print(f"  RH_liquid > RH_contr: {(result.data['rh_liquid'] > result.data['RH_contr']).sum().values}")

print(f"\nPersistent contrails: {result.data['potential_persistent_contrail'].sum().values}")

# Check a specific point where RHi > 1
idx = np.where(result.data['rhi'].values > 1.0)
if len(idx[0]) > 0:
    i = idx[0][0]
    print(f"\n=== Example point with RHi > 1 ===")
    print(f"RHi: {result.data['rhi'].values.flat[i]:.3f}")
    print(f"T: {result.data['air_temperature'].values.flat[i]:.2f} K")
    print(f"T_contr: {result.data['T_contr'].values.flat[i]:.2f} K")
    print(f"RH_liquid: {result.data['rh_liquid'].values.flat[i]:.3f}")
    print(f"RH_contr: {result.data['RH_contr'].values.flat[i]:.3f}")
    print(f"G: {result.data['G'].values.flat[i]:.6f}")



In [ ]:
print(f"Air pressure min/max: {result.data['air_pressure'].min().values} / {result.data['air_pressure'].max().values}")
print(f"Air pressure units: {result.data['air_pressure'].attrs.get('units', 'not specified')}")
print(f"Air pressure shape: {result.data['air_pressure'].shape}")
print(f"Air pressure dims: {result.data['air_pressure'].dims}")

# Also check a single value
print(f"\nSample pressure value: {result.data['air_pressure'].values.flat[0]}")

In [ ]:
from pycontrails.physics import constants
print(f"c_pd = {constants.c_pd}")
print(f"epsilon = {constants.epsilon}")

In [ ]:
print(f"air_pressure dims: {met.data['air_pressure'].dims}")
print(f"air_pressure coords: {met.data['air_pressure'].coords}")
print(f"air_temperature dims: {met.data['air_temperature'].dims}")